# Notebook 23 — Full Paper Generator

**prime-numbers-lab**

This patched generator creates a complete paper folder and fixes the LaTeX abstract / figure naming issues at the source.

In [ ]:

import json
import shutil
import zipfile
from pathlib import Path
from datetime import datetime

import pandas as pd

NOTEBOOK_ID = "23_full_paper_generator"
OUTDIR = Path(NOTEBOOK_ID)
FIGDIR = OUTDIR / "figures"
DATADIR = OUTDIR / "data"
DOCDIR = OUTDIR / "docs"
TEXDIR = OUTDIR / "tex"

PAPER_ROOT = OUTDIR / "paper"
PAPER_SECTIONS = PAPER_ROOT / "sections"
PAPER_FIGURES = PAPER_ROOT / "figures"
PAPER_TABLES = PAPER_ROOT / "tables"

for d in [OUTDIR, FIGDIR, DATADIR, DOCDIR, TEXDIR, PAPER_ROOT, PAPER_SECTIONS, PAPER_FIGURES, PAPER_TABLES]:
    d.mkdir(parents=True, exist_ok=True)

SOURCE_FIGURES_DIRS = [
    Path("figures"),
    Path("../figures"),
    Path("16_transition_operator_entropy_mixing/figures"),
    Path("17_higher_order_transition_memory_shuffle_baseline/figures"),
    Path("18_spectral_memory_low_rank_operator/figures"),
    Path("20_synthetic_controls_generalization/figures"),
    Path("21_statistical_validation_layer/figures"),
]

print("Notebook:", NOTEBOOK_ID)
print("Paper directory:", PAPER_ROOT.resolve())

In [ ]:

FIGURE_REGISTRY = [
    {
        "source_name": "16_transition_operator_heatmap.png",
        "paper_name": "fig1_transition_operator.png",
        "label": "fig:transition-operator",
        "caption": "Empirical first-order transition operator for consecutive prime residues modulo 30. Rows indicate the current residue class and columns indicate the next residue class. This operator defines the Markov baseline used throughout the analysis.",
    },
    {
        "source_name": "17_two_step_operator_delta_heatmap.png",
        "paper_name": "fig2_two_step_delta.png",
        "label": "fig:two-step-delta",
        "caption": "Two-step residual operator, defined as the empirical two-step transition operator minus the first-order Markov prediction. Structured positive and negative regions indicate higher-order transition memory beyond the first-order baseline.",
    },
    {
        "source_name": "20_real_vs_controls_singular_spectrum.png",
        "paper_name": "fig3_singular_spectrum.png",
        "label": "fig:singular-spectrum",
        "caption": "Singular value spectrum of the two-step residual operator compared with synthetic controls. The real sequence exhibits a stronger leading spectrum, consistent with low-rank structured memory.",
    },
    {
        "source_name": "21_pvalue_matrix.png",
        "paper_name": "fig4_pvalues.png",
        "label": "fig:pvalue-matrix",
        "caption": "Statistical validation matrix showing significance against null controls. Structure-destroying controls produce small p-values across multiple metrics, while block- and window-preserving controls retain local structure.",
    },
    {
        "source_name": "21_publication_validation_summary.png",
        "paper_name": "fig5_effect_sizes.png",
        "label": "fig:effect-sizes",
        "caption": "Publication summary of effect sizes, reported as real metric divided by null mean. Large ratios indicate that the observed memory structure is not explained by simple null models.",
    },
]

def find_source_figure(filename):
    for folder in SOURCE_FIGURES_DIRS:
        p = folder / filename
        if p.exists():
            return p
    for p in Path(".").rglob(filename):
        if p.is_file():
            return p
    return None

copied_figures = []
missing_figures = []

for item in FIGURE_REGISTRY:
    src = find_source_figure(item["source_name"])
    dst = PAPER_FIGURES / item["paper_name"]
    if src is None:
        missing_figures.append(item["source_name"])
        continue
    shutil.copy2(src, dst)
    copied_figures.append({**item, "source_path": str(src), "paper_path": str(dst), "size_bytes": dst.stat().st_size})

pd.DataFrame(copied_figures).to_csv(DATADIR / "23_figure_registry.csv", index=False)
pd.DataFrame(copied_figures).to_csv(PAPER_TABLES / "figure_registry.csv", index=False)
pd.DataFrame({"missing_source_name": missing_figures}).to_csv(DATADIR / "23_missing_figures.csv", index=False)

print("Copied figures:", len(copied_figures))
print("Missing figures:", missing_figures)

In [ ]:

PAPER_TITLE = "Higher-Order Residue Memory in Prime Gap Transitions"
PAPER_AUTHOR = "Dan Hawkley"
PAPER_REPO = "github.com/thinkthoughts/prime-numbers-lab"

ABSTRACT = r"""
We study residue-class transitions of consecutive primes modulo $30$ as a finite computational dynamical system. Using the first-order transition operator $P$ as a Markov baseline, we compare the empirical two-step operator $P^{(2)}$ with $P^2$ and analyze the residual $\Delta = P^{(2)} - P^2$. The residual exhibits low-rank structure, stable singular modes, and measurable predictive gains under low-rank correction. Synthetic controls---including i.i.d.\ shuffles, Markov-generated sequences, block shuffles, balanced shuffles, and gap-based shuffles---show that the observed structure is not explained by marginal residue frequencies or first-order transition statistics alone. Bootstrap and null validation demonstrate statistically significant separation from structure-destroying controls while remaining consistent with local block-preserving resampling. These results provide empirical computational evidence for higher-order residue memory in finite prime-gap transition data, without asserting an asymptotic theorem.
""".strip()

if ABSTRACT.count("$") % 2 != 0:
    raise ValueError("Abstract has unbalanced inline math delimiters.")

metadata = {
    "title": PAPER_TITLE,
    "author": PAPER_AUTHOR,
    "repo": PAPER_REPO,
    "generated": datetime.utcnow().isoformat() + "Z",
    "figure_count": len(copied_figures),
    "missing_figure_count": len(missing_figures),
}
(DATADIR / "23_paper_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
metadata

In [ ]:

sections = {
"00_abstract.tex": ABSTRACT,
"01_introduction.tex": r"""
\section{Introduction}

Prime gaps exhibit both regular arithmetic constraints and irregular statistical behavior. Consecutive primes greater than $5$ lie in the residue classes
\[
R=\{1,7,11,13,17,19,23,29\}\pmod{30}.
\]
This finite residue structure provides a natural setting for empirical transition analysis. Rather than treating prime gaps only through scalar gap sizes, we study transitions among residue classes as a finite computational dynamical system.

The first-order transition operator $P$ gives a Markov baseline for consecutive residue changes. If first-order dynamics fully explain the observed sequence, the empirical two-step operator $P^{(2)}$ should be close to $P^2$. Deviations from this baseline define a residual operator
\[
\Delta = P^{(2)}-P^2.
\]
The central question of this paper is whether $\Delta$ behaves like sampling noise or exhibits structured, statistically validated memory.
""".strip(),
"02_methods.tex": r"""
\section{Methods}

Let $p_n$ denote the $n$th prime and define the residue state
\[
r_n = p_n \bmod 30,
\]
restricted to the eight residue classes coprime to $30$. The empirical first-order transition operator is
\[
P_{ij}=\Pr(r_{n+1}=j\mid r_n=i).
\]
The empirical two-step operator is
\[
P^{(2)}_{ik}=\Pr(r_{n+2}=k\mid r_n=i).
\]
The first-order Markov prediction is $(P^2)_{ik}$, and the higher-order residual is
\[
\Delta_{ik}=P^{(2)}_{ik}-(P^2)_{ik}.
\]

We analyze $\Delta$ using singular value decomposition,
\[
\Delta=\sum_{\ell=1}^{8}\sigma_\ell u_\ell v_\ell^\top,
\]
and evaluate low-rank reconstructions
\[
\Delta_k=\sum_{\ell=1}^{k}\sigma_\ell u_\ell v_\ell^\top.
\]
Synthetic controls include i.i.d.\ shuffles, Markov-generated sequences, block shuffles, balanced shuffles, and gap-based shuffles. Validation metrics include residual norms, singular spectra, rank requirements, mutual information, p-values against empirical nulls, and windowed stability.
""".strip(),
"03_results.tex": r"""
\section{Results}

Figure~\ref{fig:transition-operator} shows the empirical first-order residue transition operator. This operator defines the baseline against which higher-order structure is measured. Figure~\ref{fig:two-step-delta} shows the two-step residual $\Delta=P^{(2)}-P^2$, revealing structured positive and negative deviations rather than uniform noise.

The spectral analysis in Figure~\ref{fig:singular-spectrum} shows that the real residual operator has stronger leading singular values than the control baselines. This indicates that the higher-order signal is concentrated in a small number of modes. Low-rank correction therefore improves prediction of the empirical two-step operator relative to the uncorrected Markov baseline.
""".strip(),
"04_validation.tex": r"""
\section{Validation}

Figure~\ref{fig:pvalue-matrix} summarizes null-model significance tests. Structure-destroying nulls, including i.i.d.\ shuffle and Markov-generated controls, separate from the real sequence across multiple metrics. Figure~\ref{fig:effect-sizes} reports real-to-null effect ratios, showing that the observed higher-order structure is substantially larger than simple null expectations.

Block- and window-preserving controls retain more of the observed structure, which is consistent with the interpretation that local ordering and finite-window constraints are part of the signal. This pattern supports the claim that the measured memory is not explained by marginal residue frequencies or first-order transition statistics alone.
""".strip(),
"05_discussion.tex": r"""
\section{Discussion}

The results support an empirical operator-level description of prime residue transitions. The first-order transition operator captures broad transition structure, while the residual $\Delta=P^{(2)}-P^2$ captures higher-order memory. The low-rank structure of $\Delta$ suggests that the memory component is not arbitrary full-dimensional noise, but is organized into a small number of dominant modes.

This work should be interpreted as a finite computational study. It does not prove an asymptotic theorem about prime gaps and does not imply a proof of any prime-number conjecture. Its contribution is a reproducible empirical framework for measuring higher-order residue memory and comparing it against explicit controls.
""".strip(),
"06_limitations.tex": r"""
\section{Limitations}

The analysis is finite-range and depends on selected computational parameters, including maximum prime bound, residue modulus, windowing, and discretization choices. Although the controls reduce the risk of spurious conclusions, they do not exhaust all possible null hypotheses. The term ``memory'' is used operationally to describe higher-order statistical dependence relative to a first-order transition baseline, not as a claim about a generative mechanism for primes.
""".strip(),
"07_conclusion.tex": r"""
\section{Conclusion}

We identify statistically validated, low-rank two-step structure in residue-class transitions of consecutive prime gaps, relative to first-order Markov and shuffled controls. The results provide empirical computational evidence that finite prime-gap transition data contain higher-order residue memory beyond marginal residue frequencies and first-order transition statistics.
""".strip(),
}

for filename, content in sections.items():
    (PAPER_SECTIONS / filename).write_text(content + "\n", encoding="utf-8")

section_registry = pd.DataFrame([{"section_file": filename, "chars": len(content)} for filename, content in sections.items()])
section_registry.to_csv(DATADIR / "23_section_registry.csv", index=False)
section_registry.to_csv(PAPER_TABLES / "section_registry.csv", index=False)
section_registry

In [ ]:

figure_blocks = []
for item in copied_figures:
    figure_blocks.append(rf"""
\begin{{figure}}[ht]
\centering
\includegraphics[width=0.92\linewidth]{{figures/{item['paper_name']}}}
\caption{{{item['caption']}}}
\label{{{item['label']}}}
\end{{figure}}
""".strip())

(PAPER_SECTIONS / "08_figures.tex").write_text("\n\n".join(figure_blocks) + "\n", encoding="utf-8")

(PAPER_ROOT / "references.bib").write_text(r"""
@misc{repo,
  author = {Hawkley, Dan},
  title = {Prime Numbers Lab},
  howpublished = {\url{https://github.com/thinkthoughts/prime-numbers-lab}},
  year = {2026}
}
""".strip() + "\n", encoding="utf-8")

(PAPER_ROOT / "Makefile").write_text(r"""
.PHONY: all clean

all:
	latexmk -pdf main.tex

fallback:
	pdflatex main.tex
	pdflatex main.tex

clean:
	latexmk -c
	rm -f *.aux *.log *.out *.toc *.fls *.fdb_latexmk *.bbl *.blg
""".strip() + "\n", encoding="utf-8")

main_tex = rf"""
\documentclass[11pt]{{article}}

\usepackage[margin=1in]{{geometry}}
\usepackage{{amsmath,amssymb}}
\usepackage{{graphicx}}
\usepackage{{booktabs}}
\usepackage{{hyperref}}
\usepackage{{url}}

\title{{{PAPER_TITLE}}}
\author{{{PAPER_AUTHOR}\\
\small \url{{https://{PAPER_REPO}}}}}
\date{{\today}}

\begin{{document}}

\maketitle

\begin{{abstract}}
\input{{sections/00_abstract.tex}}
\end{{abstract}}

\input{{sections/01_introduction.tex}}
\input{{sections/02_methods.tex}}
\input{{sections/03_results.tex}}
\input{{sections/04_validation.tex}}
\input{{sections/05_discussion.tex}}
\input{{sections/06_limitations.tex}}
\input{{sections/07_conclusion.tex}}
\input{{sections/08_figures.tex}}

\bibliographystyle{{plain}}
\bibliography{{references}}

\end{{document}}
""".strip()

(PAPER_ROOT / "main.tex").write_text(main_tex + "\n", encoding="utf-8")
print("Wrote", PAPER_ROOT / "main.tex")

In [ ]:

checks = {
    "abstract_dollar_count": ABSTRACT.count("$"),
    "abstract_dollars_balanced": ABSTRACT.count("$") % 2 == 0,
    "main_tex_exists": (PAPER_ROOT / "main.tex").exists(),
    "makefile_exists": (PAPER_ROOT / "Makefile").exists(),
    "figures_copied": len(copied_figures),
    "figures_missing": len(missing_figures),
}

if not checks["abstract_dollars_balanced"]:
    raise ValueError("Abstract has unbalanced $ delimiters.")

summary_md = f"""# Notebook 23 Paper Generator Summary

## Paper

- Title: {PAPER_TITLE}
- Author: {PAPER_AUTHOR}
- Repo: {PAPER_REPO}

## Build readiness

- Abstract math delimiters balanced: {checks["abstract_dollars_balanced"]}
- Figures copied: {checks["figures_copied"]}
- Missing figures: {checks["figures_missing"]}

## Curated figures

""" + "\n".join([f"- `{row['paper_name']}` from `{row['source_name']}`" for row in copied_figures])

if missing_figures:
    summary_md += "\n\n## Missing source figures\n\n" + "\n".join([f"- `{x}`" for x in missing_figures]) + "\n"

(DOCDIR / "23_paper_generator_summary.md").write_text(summary_md, encoding="utf-8")
pd.DataFrame([checks]).to_csv(DATADIR / "23_build_readiness_checks.csv", index=False)
print(summary_md)

In [ ]:

manifest_rows = []
for subdir in [FIGDIR, DATADIR, DOCDIR, TEXDIR, PAPER_ROOT]:
    for path in sorted(subdir.rglob("*")):
        if path.is_file():
            manifest_rows.append({
                "notebook_id": NOTEBOOK_ID,
                "relative_path": str(path),
                "folder": path.parent.name,
                "filename": path.name,
                "size_bytes": path.stat().st_size,
            })

manifest = pd.DataFrame(manifest_rows)
manifest.to_csv(DATADIR / "23_outputs_manifest.csv", index=False)

EXPORT_ZIP = Path(f"{NOTEBOOK_ID}_export.zip")
with zipfile.ZipFile(EXPORT_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(OUTDIR.rglob("*")):
        if path.is_file():
            zf.write(path, arcname=str(path))

print("Export zip created:", EXPORT_ZIP)
print("Files in manifest:", len(manifest))
print("Zip size bytes:", EXPORT_ZIP.stat().st_size)

# Optional: download outputs bundle (template standard)
# from google.colab import files
# files.download("23_full_paper_generator_export.zip")